# Deep Agents and Voice

*A realtime voice assistant that hands the hard questions to a deep agent.*

You talk; it talks back. For small talk it just answers. But when you ask something that needs real
research, it delegates to a **deep agent** &mdash; planning, web search, report writing &mdash; and then
narrates the findings back to you, conversationally.

This is the notebook-native retelling of
[**langchain-ai/google-adk-realtime-deepagents-example**](https://github.com/langchain-ai/google-adk-realtime-deepagents-example),
which wires the same idea into a browser app with Google ADK, FastAPI, and WebSocket audio. Here we drop
the web stack and run every piece in the kernel, so you can watch each part do its job.

<img src="images/deep-agent-voice-flow.svg"
     alt="You speak into Gemini Live, which answers small talk directly and calls a NON_BLOCKING deep_research tool for anything needing facts. A deep agent coordinator plans with write_todos and delegates to a researcher subagent that owns the Tavily search tool; the coordinator writes a short spoken report, which returns to Gemini Live to be narrated aloud."
     style="width:100%; height:auto; border:1px solid #d0d7de; border-radius:6px;" />

The voice layer is **Gemini Live** &mdash; bidirectional audio with server-side voice-activity detection,
so you just talk and it knows when you have stopped. The research brain is a **deepagents** agent backed
by **Claude** and **Tavily**, exposed to the voice model as a single `deep_research` tool.

> **Run this locally.** Live mic capture needs a real microphone and speakers, so use a local kernel
> (not a remote/Colab one). macOS will ask for microphone permission the first time you start the loop.

In [62]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Setup

Load the environment (`GEMINI_API_KEY` for the voice layer, `ANTHROPIC_API_KEY` and `TAVILY_API_KEY`
for the research brain), import what we need, and pick the two models.

In [56]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

import asyncio
from google import genai
from google.genai import types
from langchain_core.tools import tool
from tavily import TavilyClient
from deepagents import create_deep_agent

from util import (
    MIC_RATE,
    LatestJob,
    MicInput,
    SpeakerOutput,
    VoiceUI,
    run_until_stopped,
    start_session,
    stream_report,
)

model = "claude-sonnet-5"                      # the research brain (Anthropic)
LIVE_MODEL = "gemini-3.1-flash-live-preview"   # the voice layer (Gemini Live)

# Point the voice client straight at Google. If a LangChain LLM gateway is configured
# (GOOGLE_GEMINI_BASE_URL), it proxies REST fine but NOT the Live WebSocket — so a
# default client would fail the handshake with HTTP 403. Passing base_url explicitly
# bypasses the gateway for Gemini Live; Claude and Tavily below are unaffected.
# We also pass the key explicitly so an ambient GOOGLE_API_KEY can't shadow GEMINI_API_KEY.
client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY") or os.environ["GOOGLE_API_KEY"],
    http_options=types.HttpOptions(base_url="https://generativelanguage.googleapis.com/"),
)

# Model ids for Live move fast. If connecting errors with "model not found", list the
# current ones — `[m.name for m in client.models.list() if "live" in m.name]` — and swap
# LIVE_MODEL. "gemini-2.5-flash-native-audio-preview-09-2025" is one alternative.

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


# The research brain

Before any audio, let's build the part that does the actual work. A deep agent comes with planning
(`write_todos`), a virtual filesystem, and subagents out of the box &mdash; and here we lean into two of
those on purpose. The **coordinator** plans the work as a todo list, then **delegates** the searching to
a `researcher` subagent that owns the Tavily tool. That structure isn't just tidy: it gives us concrete,
streamable activity &mdash; a plan, a hand-off, each search &mdash; to put on screen while the caller
waits. Both prompts are tuned for **spoken** answers: short, plain sentences, no markdown or URLs to read
aloud.

In [ ]:
# @tool
# def internet_search(query: str, max_results: int = 5) -> list[dict]:
#     """Search the web for information.

#     Args:
#         query: what to search for
#         max_results: how many results to return
#     """
#     response = TavilyClient().search(query, max_results=max_results)
#     return response.get("results", [])


# A subagent that owns the searching. Giving the Tavily tool ONLY to the subagent means
# the coordinator can't search on its own — it has to plan and then delegate. That's the
# machinery we want on screen: a todo plan, a hand-off, and the searches streaming in
# while the caller waits for an answer. The search cap keeps a live demo snappy (and the
# activity panel readable) — a broad question can otherwise trigger a dozen-plus searches.
researcher = {
    "name": "researcher",
    "description": "Searches the web on a focused question and returns concise findings.",
    "system_prompt": (
        "You are a focused web researcher. Run at most 4-5 targeted searches — no more — "
        "cross-check claims across sources, then stop and return concise findings in plain "
        "sentences. No markdown or URLs."
    ),
    # "tools": [internet_search], # TODO replace tavily tool with tools=[{"type": "web_search"}]
}

RESEARCH_INSTRUCTIONS = """You are a research coordinator answering questions that will be READ ALOUD.

ALWAYS start by calling write_todos with a short plan (2-4 steps) — every run, even an easy one, and
before any other tool call. The plan is shown live to a waiting listener, so skipping it leaves them
staring at an empty panel. Then delegate the searching to the
`researcher` subagent with the task tool, giving it complete, self-contained instructions in one call.
Update the todos as the work progresses. When the findings come back, write a short spoken report:
3-5 sentences of plain, conversational language. No markdown, no bullet lists, no URLs. Lead with the
answer; name a source only when it genuinely matters."""

research_agent = create_deep_agent(
    model=model,
    system_prompt=RESEARCH_INSTRUCTIONS,
    subagents=[researcher],
)

The voice loop calls this agent through **`stream_report`** (in `util/pretty.py`), which returns just the
final report as plain text &mdash; ready to be spoken, with no markdown or thinking blocks in it. Instead
of a single `ainvoke` it **streams** the run with `subgraphs=True` and pushes every step to the live
activity panel: the coordinator's todo plan, the hand-off to the `researcher` subagent, and each web
search. Coordinator events arrive with an empty namespace `ns`; the subagent's arrive under a nested one,
which is how each step is attributed to the right actor &mdash; the same trick `print_activity` uses in
the basics notebook. The payoff is that there's something to watch while the research runs in the background.

# Wrapping research as a Live tool

The Live API doesn't auto-run your Python functions the way the regular API can &mdash; you **declare**
the tool as a schema, and when the model decides to call it you run it yourself and send the result back.
So we describe `deep_research` to Gemini Live as a function that takes a `topic`.

**Blocking vs. non-blocking.** By default a tool call is *blocking*: the model goes quiet until we return
the report, and our receive loop is stuck waiting on the deep agent. For a call that takes a minute that
is a poor fit &mdash; you cannot interrupt, and nothing is draining the WebSocket while you wait (long
enough and the connection dies of a keepalive timeout). So we declare it
**`behavior=NON_BLOCKING`**, like the source repo does: the model keeps the conversation going while
research runs, and we hand the report back whenever it is ready. The `research()` task in the loop cell
below is the other half of that deal.

We declare a second function alongside it: **`end_conversation`**, which the model calls when you say
goodbye. That could have been a string match on the transcript, but a tool is both sturdier and less
code &mdash; the model already knows the difference between *"goodbye, thanks!"* and *"don't say goodbye
yet"*, and routing it through a tool call means the assistant says its farewell before we close the audio
devices. Both declarations go in one `types.Tool`.

In [58]:
deep_research = types.FunctionDeclaration(
    name="deep_research",
    behavior=types.Behavior.NON_BLOCKING,
    description=(
        "Run in-depth web research on a topic and return a short, spoken-friendly report. "
        "Use whenever a good answer needs current facts, figures, comparisons, or multiple sources."
    ),
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "topic": types.Schema(
                type=types.Type.STRING,
                description="the question or topic to research",
            )
        },
        required=["topic"],
    ),
)

# Hanging up is a tool too. Letting the model decide beats matching words in the
# transcript: it can tell "goodbye, thanks!" from "don't say goodbye yet", and it gets
# to say a farewell before we close the devices. No parameters — it is just a signal.
end_conversation = types.FunctionDeclaration(
    name="end_conversation",
    behavior=types.Behavior.NON_BLOCKING,
    description=(
        "End the session and hang up. Call this as your final action once the user signals "
        "they are finished — 'bye', 'goodbye', 'that's all', 'we're done', 'talk later'. "
        "Say a short farewell first; do not call it while the user is still asking things."
    ),
)

voice_tools = types.Tool(function_declarations=[deep_research, end_conversation])

# The voice layer &mdash; Gemini Live

Now the voice session config. Gemini Live streams audio both ways: we send **16 kHz** PCM from the mic,
it sends **24 kHz** PCM back. Server-side voice-activity detection decides when you've finished speaking,
and turning on input/output transcription gives us live captions to print. The system prompt tells the
model to stay brief, answer small talk directly, and reach for `deep_research` when a question needs it.

We also set the **voice-activity timings** explicitly. The server decides your turn has ended once it
hears `SILENCE_MS` of quiet; until then it is still listening, not thinking. That wait is the first of
three things standing between you and a reply &mdash; the other two being the model's own time to first
audio, and however much buffering the sound card adds (`util/voice.py` asks CoreAudio for low-latency
buffers instead of sounddevice's safe default). The `⚡` line in the transcript reports the first two
together, so you can tell a slow model from a slow microphone.

In [59]:
VOICE_INSTRUCTIONS = """You are a warm, concise voice assistant. Keep replies short and natural —
you are being heard, not read.

You are NOT a general-knowledge oracle. For ANY question involving facts, current events, numbers,
comparisons, or anything described as latest/recent — or anything you can't answer with complete
confidence from memory — you MUST call the deep_research tool with a clear topic instead of answering
from your own knowledge. Only answer directly for pure small talk (greetings, chit-chat, clarifying
questions).

When you call deep_research, give a brief spoken acknowledgement first (like "Sure, let me look into
that"). The research runs in the background, so stay available while it does: answer follow-ups, and if
the user asks for something different, just call deep_research again with the new topic. When a report
comes back, weave it into a natural spoken answer and offer to go deeper.

When the user signals they are done — "bye", "goodbye", "that's all", "thanks, we're finished" — say a
short, warm farewell and then call end_conversation as your final action. Do not call it while they are
still asking for things, and never call it just because a topic wrapped up."""

# How long the server waits, after you stop making noise, before it decides your turn
# is over and starts generating. This is usually the biggest slice of the gap between
# "Hi" and hearing anything back, and the default is conservative. Shorten it and the
# assistant feels snappier; shorten it too far and it cuts in during your pauses.
SILENCE_MS = 400   # end-of-speech wait
PREFIX_MS = 120    # speech needed before a turn is considered started

live_config = types.LiveConnectConfig(
    system_instruction=VOICE_INSTRUCTIONS,
    response_modalities=["AUDIO"],
    tools=[voice_tools],
    realtime_input_config=types.RealtimeInputConfig(
        automatic_activity_detection=types.AutomaticActivityDetection(
            start_of_speech_sensitivity=types.StartSensitivity.START_SENSITIVITY_HIGH,
            end_of_speech_sensitivity=types.EndSensitivity.END_SENSITIVITY_HIGH,
            prefix_padding_ms=PREFIX_MS,
            silence_duration_ms=SILENCE_MS,
        )
    ),
    input_audio_transcription=types.AudioTranscriptionConfig(),
    output_audio_transcription=types.AudioTranscriptionConfig(),
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
        )
    ),
)

# Audio and notebook plumbing

Capturing the mic, playing audio back, driving the widgets, and minding background tasks is all
boilerplate, so it lives in `util/voice.py`:

- **`MicInput`** opens a 16 kHz input stream and hands each PCM frame to asyncio, coalesced into 20 ms
  chunks. `frames(mute_while=...)` drops frames while a predicate is true &mdash; the half-duplex gate.
- **`SpeakerOutput`** plays the model's 24 kHz PCM gaplessly. `flush()` drops queued audio for
  **barge-in** on headphones, `is_speaking()` reports whether sound is still coming out (it measures the
  *playback* queue, not arrival time &mdash; Live delivers a turn in a burst, so several seconds of
  speech can land in under one), and `drain()` waits for a farewell to finish.
- **`VoiceUI`** owns the widgets: the Stop button, the live captions (transcripts arrive in fragments, so
  it buffers and attributes them), the `LiveActivityPanel`, and the optional `⚡` turnaround readout.
- **`LatestJob`** runs at most one background job, cancelling any predecessor &mdash; the bookkeeping that
  keeps one research run in flight without leaking tasks.
- **`run_until_stopped`** runs the two pumps until Stop or the time cap, then cancels and awaits them. A
  coroutine that *returns* does not end the session; one that *raises* does, and the error is re-raised.
- **`start_session`** runs the whole session as a background task, which is what makes the Stop button
  work at all &mdash; see below.

`MicInput` and `SpeakerOutput` are context managers, so the device streams always close cleanly however
the loop ends.

# The realtime loop

This is the whole thing. Inside one Live session we run two coroutines at once:

- **uplink** &mdash; forward mic frames to the model (muted while the assistant is speaking &mdash; see below).
- **downlink** &mdash; for each message: play audio, caption transcripts, handle barge-in, and when the
  model calls `deep_research`, kick off a **background `research()` task** &mdash; which streams each step
  into the live activity panel so you can watch it plan, delegate, and search, then hands the report back
  to be narrated.

> **One `receive()` is one turn.** The SDK ends the `session.receive()` iterator the moment the server
> sets `turn_complete`, so downlink wraps it in a `while` loop and re-enters it for the next turn. Miss
> that and the assistant answers your first question and then goes silent &mdash; the captions for later
> turns never arrive, because nothing is reading the socket any more. A *tool call*, by contrast, does
> **not** end the turn: we answer it with `send_tool_response` and the same iterator carries the narrated
> report back.

> **Interrupting research.** The reason `research()` is a separate task is that `await`-ing the deep
> agent inline would freeze downlink for the whole minute: no captions, no barge-in, and nothing reading
> the socket &mdash; long enough and the WebSocket dies of a keepalive ping timeout. As its own task the
> agent works while downlink keeps reading, so you can talk over the assistant mid-research and ask for
> something else. A new topic **supersedes** the old one: the in-flight run is cancelled, its abandoned
> tool call is closed out with `will_continue=False` + `SILENT` so the model stops waiting on it, and the
> panel switches to the new topic. Want both instead? Keep a task per call rather than one, and give each
> its own panel.

Below the transcript sits the **activity panel**: the moment research starts it shows the coordinator's
todo plan, the hand-off to the `researcher` subagent, and every web search as it fires, with a ticking
`⏳ Ns` timer so there's always motion &mdash; no more silent dead air while you wait. A **Stop** button
(and a safety time cap) end the session cleanly.

> **Why you can't talk over it.** One microphone hears both you and the assistant, so streaming the
> mic while the assistant speaks feeds its own voice back: the server transcribes the agent as *you*,
> decides you are interrupting, and clips the reply — over and over. A loudness threshold cannot separate
> the two, because "user speaking loudly" and "assistant playing loudly" are the same signal. Real
> barge-in on speakers needs acoustic echo cancellation. So we run **half-duplex**: the mic is muted
> while the assistant speaks (`mute_while=speaker.is_speaking`) and you wait for it to finish. Note that
> the gate has to follow the *speakers*, not the socket &mdash; Live sends a turn's audio in a burst, so
> gating on "audio arrived recently" opens the mic seconds before playback actually ends. On
> headphones there is no echo path at all — drop `mute_while` and you get true barge-in for free.
>
> You *can* still redirect mid-research, though: while the deep agent works the assistant is quiet, so
> the mic is open. Ask for a different topic and the in-flight run is superseded.

> **Hanging up.** Say goodbye and the model calls `end_conversation`; we acknowledge it `SILENT` (so it
> does not say goodbye twice), let the farewell drain out of the speaker, and set the stop event. The
> hang-up happens *after* `receive()` ends, which is the point where the turn is complete and all of the
> farewell audio is queued — stopping the moment the tool call arrives would clip it mid-word.

> **Why the session runs in the background.** A notebook cell sitting in `await` stops the kernel from
> handling any further shell messages, and widget callbacks arrive as shell messages &mdash; so a
> `Stop` button clicked during `await voice_session()` is not merely slow, it is queued until the cell
> ends, which is exactly when you no longer need it. So the cell doesn't await: `start_session` puts the
> session on the event loop and returns the task, leaving the kernel free to deliver clicks while the
> loop drives the session. Re-running the cell supersedes the previous session rather than leaving two
> of them fighting over the microphone, and because a detached task's exception would otherwise vanish
> silently, `start_session` reports it. Three ways out, then: the button, saying goodbye, or
> `session.cancel()`.

**How to use it:** run the cell, allow microphone access when macOS asks, then talk. Small talk
(*"how's it going?"*) is answered directly. A research question
(*"what's the latest on solid-state EV batteries?"*) lights up the activity panel &mdash; plan &rarr;
delegate &rarr; searches, ticking away &mdash; then you hear the spoken report. While it researches you
can ask for a different topic and it will switch. Say goodbye to end the session, or click **Stop**.
The cell returns straight away &mdash; that is deliberate; the session keeps running in the background.

In [60]:
MAX_SECONDS = 180  # safety cap so the session always ends

ui = VoiceUI(show_latency=True)  # Stop button, captions, activity panel, ⚡ turnaround


async def voice_session():
    ui.reset()
    research = LatestJob()  # one research run at a time; a new topic supersedes it

    async with client.aio.live.connect(model=LIVE_MODEL, config=live_config) as session:
        with MicInput() as mic, SpeakerOutput() as speaker:

            async def uplink():
                # Half-duplex: the mic goes deaf while the assistant is speaking, so the
                # speakers can't echo back and be mistaken for you interrupting.
                # is_speaking() tracks audio still *playing*, not audio that merely
                # arrived — Live sends a turn in a burst, so those are seconds apart.
                # On headphones, drop `mute_while` for true barge-in.
                async for frame in mic.frames(mute_while=speaker.is_speaking):
                    await session.send_realtime_input(
                        audio=types.Blob(data=frame, mime_type=f"audio/pcm;rate={MIC_RATE}")
                    )

            async def research_and_report(fc, topic):
                # A background job, so downlink stays free to read the socket: that is
                # what lets you talk while the deep agent works, and what keeps the
                # WebSocket drained — a minute-long blocking call kills the connection.
                report = await stream_report(research_agent, topic, ui.activity)
                await session.send_tool_response(function_responses=[types.FunctionResponse(
                    id=fc.id, name=fc.name, response={"report": report},
                    # INTERRUPT narrates the findings as soon as they land;
                    # WHEN_IDLE would wait for a natural pause instead.
                    scheduling=types.FunctionResponseScheduling.INTERRUPT,
                )])

            async def close_silently(fc):
                # Answer a call without prompting speech: will_continue=False ends it,
                # SILENT stops it triggering generation. Used to drop a superseded
                # research call, and to acknowledge a hang-up without a second goodbye.
                await session.send_tool_response(function_responses=[types.FunctionResponse(
                    id=fc.id, name=fc.name, response={}, will_continue=False,
                    scheduling=types.FunctionResponseScheduling.SILENT,
                )])

            async def downlink():
                hanging_up = False
                # session.receive() covers ONE turn: the SDK ends the iterator as soon as
                # the server sets turn_complete. Re-enter it to keep the conversation
                # going, or the assistant answers once and then goes quiet.
                while not ui.stopped.is_set():
                    async for msg in session.receive():
                        if msg.data:
                            ui.note_reply_audio()              # ⚡ time to first audio
                            speaker.play(msg.data)             # model audio → speakers

                        if msg.server_content:
                            if msg.server_content.interrupted:
                                speaker.flush()               # barge-in: drop stale speech
                            ui.caption(msg.server_content)    # live transcripts

                        for fc in (msg.tool_call.function_calls if msg.tool_call else []):
                            if fc.name == "end_conversation":
                                hanging_up = True
                                await close_silently(fc)
                                continue

                            topic = (fc.args or {}).get("topic", "")
                            ui.researching(topic)
                            superseded = await research.start(
                                research_and_report(fc, topic), key=fc
                            )
                            if superseded is not None:
                                await close_silently(superseded)

                    # receive() just ended, so the farewell turn is complete and all of
                    # its audio is queued: let it play out, then close up.
                    if hanging_up:
                        ui.hanging_up()
                        await speaker.drain()
                        ui.stopped.set()

            try:
                await run_until_stopped(uplink, downlink, stop=ui.stopped, timeout=MAX_SECONDS)
            finally:
                research.cancel()  # a detached job dies with its session


# Not awaited: a cell parked on `await` blocks the kernel from delivering widget
# events, which is what stops the Stop button working. Cancel with the button, by
# saying goodbye, or with `session.cancel()`.
session = await start_session(voice_session(), ui)


Button(button_style='danger', description='Stop', icon='stop', style=ButtonStyle())

HTML(value='')

HTML(value='')

# Recap

- **A realtime voice model with delegation to a deep agent** driven straight from `google-genai`'s Live API.
- **The deep agent is just a tool.** We declared `deep_research` to Gemini Live, and on a tool call ran a
  `create_deep_agent` (Claude + Tavily) and streamed the report back for the model to narrate.
- **Non-blocking tool calls.** `deep_research` is declared `NON_BLOCKING` and runs as a background task,
  so the voice keeps flowing, you can interrupt mid-research, and the socket stays drained. The report
  comes back with `scheduling=INTERRUPT`; a superseded call is closed out with `will_continue=False`.